<a href="https://colab.research.google.com/github/irfan-ardiansyah/data-science-2026/blob/main/Pertemuan_12_Muhammad_Irfan_Ardiansyah_250401020175.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
  n_item = np.random.randint(2, 6)
  transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
  if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
    transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [19]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [20]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
  print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


In [21]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

         antecedents consequents  support  confidence      lift
8        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
13  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
9       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
15     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
10     (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
14   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


## Interpretasi Association Rules

Aturan dengan nilai Lift tertinggi adalah:

**{Teh, Keju} → {Telur}** dengan nilai Lift sebesar **2.38**.

Nilai Confidence sebesar **85.71%** menunjukkan bahwa dari seluruh transaksi yang mengandung Teh dan Keju, sebanyak 85.71% juga mengandung Telur.

Nilai Lift yang lebih besar dari 1 menunjukkan adanya hubungan positif yang cukup kuat antara ketiga produk tersebut. Pelanggan yang membeli Teh dan Keju memiliki kemungkinan sekitar 2.38 kali lebih besar untuk membeli Telur dibandingkan pelanggan secara umum.

Secara bisnis, aturan ini masih cukup masuk akal meskipun tidak seintuitif pasangan seperti Roti dan Selai. Kemungkinan pola ini muncul karena dataset yang digunakan merupakan dataset sintetis yang dibuat secara acak dengan beberapa pola tersembunyi yang disisipkan selama proses pembangkitan data.

Sebagai perbandingan, aturan **Roti → Selai** juga ditemukan dengan Lift sebesar **1.32**, yang lebih mudah dijelaskan secara bisnis karena kedua produk tersebut memang sering dikonsumsi bersamaan dan umum digunakan dalam strategi cross-selling maupun bundling produk.

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
  idx = katalog.index[katalog['produk'] == nama_produk][0]

  skor = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]

  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [23]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung

produk_target
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


## Perbandingan Association Rules dan Content-Based Filtering

Untuk produk target **Roti**, metode Association Rules merekomendasikan produk **Selai** dengan nilai Lift sebesar **1.92** dan **1.32** pada dua aturan yang berbeda. Nilai Lift yang lebih besar dari 1 menunjukkan adanya hubungan positif antara Roti dan Selai, sehingga pelanggan yang membeli Roti memiliki kemungkinan lebih besar untuk membeli Selai dibandingkan pelanggan secara umum.

Sementara itu, metode Content-Based Filtering merekomendasikan produk **Selai, Sereal, dan Susu** berdasarkan kesamaan karakteristik atau kategori produk.

Kedua pendekatan memberikan hasil yang cukup konsisten karena sama-sama merekomendasikan **Selai** sebagai produk yang relevan untuk Roti. Hal ini menunjukkan bahwa Selai tidak hanya memiliki kemiripan karakteristik dengan Roti, tetapi juga sering dibeli bersamaan oleh pelanggan.

Association Rules lebih cocok digunakan untuk:
- Menemukan pola pembelian pelanggan.
- Cross-selling dan bundling produk.
- Rekomendasi "Pelanggan yang membeli ini juga membeli...".

Content-Based Filtering lebih cocok digunakan untuk:
- Merekomendasikan produk dengan karakteristik serupa.
- Mengatasi masalah cold start pada pengguna atau produk baru.
- Situasi ketika data transaksi historis masih terbatas.

Dalam implementasi nyata seperti pada e-commerce modern, kedua pendekatan biasanya digabungkan menjadi sistem rekomendasi hybrid untuk menghasilkan rekomendasi yang lebih akurat dan relevan bagi pengguna.